# 04 — Human Review State Machine Demo

Implements Chapter 4's `DRAFTED -> UNDER_REVIEW -> APPROVED/REJECTED -> FINALIZED` state machine as runnable code, with an append-only audit-trail log recording who changed what and when, and the `REJECTED` -> regenerate path that retains the prior draft rather than overwriting it.

In [1]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum

class NarrativeState(Enum):
    DRAFTED = 'DRAFTED'
    UNDER_REVIEW = 'UNDER_REVIEW'
    APPROVED = 'APPROVED'
    REJECTED = 'REJECTED'
    FINALIZED = 'FINALIZED'

# Legal transitions, per Chapter 4's state machine diagram
ALLOWED_TRANSITIONS = {
    NarrativeState.DRAFTED: {NarrativeState.UNDER_REVIEW},
    NarrativeState.UNDER_REVIEW: {NarrativeState.APPROVED, NarrativeState.REJECTED},
    NarrativeState.APPROVED: {NarrativeState.FINALIZED},
    NarrativeState.REJECTED: set(),  # terminal for THIS draft; triggers a NEW DRAFTED narrative
    NarrativeState.FINALIZED: set(),  # terminal, immutable
}

@dataclass
class AuditEvent:
    narrative_id: str
    from_state: str
    to_state: str
    actor: str
    timestamp: datetime
    reason: str = None

# Append-only log -- 'stored as an append-only log rather than mutable fields on the
# narrative record itself, so the audit trail can't be silently altered after the fact.'
AUDIT_LOG: list = []

print('State machine and audit log initialized.')

State machine and audit log initialized.


In [2]:
class Narrative:
    def __init__(self, narrative_id: str, prior_draft_id: str = None):
        self.narrative_id = narrative_id
        self.state = NarrativeState.DRAFTED
        self.prior_draft_id = prior_draft_id  # linked, not overwritten, on regenerate

    def transition(self, to_state: NarrativeState, actor: str, reason: str = None, when: datetime = None):
        if to_state not in ALLOWED_TRANSITIONS[self.state]:
            raise ValueError(
                f"Illegal transition: {self.state.value} -> {to_state.value} "
                f"for narrative {self.narrative_id}"
            )
        when = when or datetime.utcnow()
        AUDIT_LOG.append(AuditEvent(self.narrative_id, self.state.value, to_state.value, actor, when, reason))
        self.state = to_state
        # APPROVED -> FINALIZED is a SYSTEM action, distinct from the human APPROVED action,
        # per Chapter 4's explicit defense of keeping these two states separate.
        if to_state == NarrativeState.APPROVED:
            self.transition(NarrativeState.FINALIZED, actor='system', reason='auto-finalize on approval', when=when)

n1 = Narrative('NARR-001')
print(f"n1 created in state: {n1.state.value}")

n1.transition(NarrativeState.UNDER_REVIEW, actor='system', reason='assigned to queue')
print(f"n1 -> {n1.state.value}")

try:
    n1.transition(NarrativeState.FINALIZED, actor='officer_jsmith')
except ValueError as e:
    print(f"Correctly rejected illegal transition: {e}")

n1 created in state: DRAFTED
n1 -> UNDER_REVIEW
Correctly rejected illegal transition: Illegal transition: UNDER_REVIEW -> FINALIZED for narrative NARR-001


C:\Users\abhis\AppData\Local\Temp\ipykernel_16276\348254276.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  when = when or datetime.utcnow()


## The approve path — APPROVED and FINALIZED as two distinct, separately-timestamped events

In [3]:
n1.transition(NarrativeState.APPROVED, actor='officer_jsmith', reason='approved with minor edit to red flags section')
print(f"n1 final state: {n1.state.value}")

print('\nAudit trail for NARR-001:')
for event in AUDIT_LOG:
    if event.narrative_id == 'NARR-001':
        print(f"  {event.timestamp.isoformat()}  {event.from_state:>12} -> {event.to_state:<12} by {event.actor}  ({event.reason})")

approve_event = next(e for e in AUDIT_LOG if e.narrative_id == 'NARR-001' and e.to_state == 'APPROVED')
finalize_event = next(e for e in AUDIT_LOG if e.narrative_id == 'NARR-001' and e.to_state == 'FINALIZED')
assert approve_event.actor == 'officer_jsmith'
assert finalize_event.actor == 'system'
print("\nPASS: APPROVED is logged as the officer's action; FINALIZED is a separate, system-attributed event.")

n1 final state: FINALIZED

Audit trail for NARR-001:
  2026-07-28T17:49:12.595045       DRAFTED -> UNDER_REVIEW by system  (assigned to queue)
  2026-07-28T17:49:12.600848  UNDER_REVIEW -> APPROVED     by officer_jsmith  (approved with minor edit to red flags section)
  2026-07-28T17:49:12.600848      APPROVED -> FINALIZED    by system  (auto-finalize on approval)

PASS: APPROVED is logged as the officer's action; FINALIZED is a separate, system-attributed event.


C:\Users\abhis\AppData\Local\Temp\ipykernel_16276\348254276.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  when = when or datetime.utcnow()


## The reject-and-regenerate path — prior draft retained, never deleted

In [4]:
n2 = Narrative('NARR-002')
n2.transition(NarrativeState.UNDER_REVIEW, actor='system', reason='assigned to queue')
n2.transition(NarrativeState.REJECTED, actor='officer_avargas',
              reason='historical-context section too thin -- retrieval only found 0 prior cases, re-run with wider case-note filter')
print(f"n2 final state: {n2.state.value} (terminal for this draft)")

# Regeneration creates a NEW narrative, linked to the rejected one -- n2 itself is never
# deleted or overwritten.
n2_regenerated = Narrative('NARR-002-R1', prior_draft_id='NARR-002')
print(f"Regenerated narrative {n2_regenerated.narrative_id} created, linked to prior draft {n2_regenerated.prior_draft_id}")

print('\nFull audit trail for the NARR-002 lineage:')
for event in AUDIT_LOG:
    if event.narrative_id == 'NARR-002':
        print(f"  {event.timestamp.isoformat()}  {event.from_state:>12} -> {event.to_state:<12} by {event.actor}  ({event.reason})")

assert n2.state == NarrativeState.REJECTED, 'The original rejected draft must remain, in REJECTED state, not be deleted.'
assert n2_regenerated.prior_draft_id == 'NARR-002', 'The regenerated narrative must link back to what it superseded.'
print("\nPASS: rejected draft NARR-002 still exists in the system (state=REJECTED, not deleted),")
print("      and the new attempt NARR-002-R1 is explicitly linked to it -- both are part of the auditable history.")

n2 final state: REJECTED (terminal for this draft)
Regenerated narrative NARR-002-R1 created, linked to prior draft NARR-002

Full audit trail for the NARR-002 lineage:
  2026-07-28T17:49:12.606992       DRAFTED -> UNDER_REVIEW by system  (assigned to queue)
  2026-07-28T17:49:12.607027  UNDER_REVIEW -> REJECTED     by officer_avargas  (historical-context section too thin -- retrieval only found 0 prior cases, re-run with wider case-note filter)

PASS: rejected draft NARR-002 still exists in the system (state=REJECTED, not deleted),
      and the new attempt NARR-002-R1 is explicitly linked to it -- both are part of the auditable history.


C:\Users\abhis\AppData\Local\Temp\ipykernel_16276\348254276.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  when = when or datetime.utcnow()
